# Semantic Similarity Detection
## Using Sentence Transformers, Cosine Similarity, and Sentiment Analysis

This notebook demonstrates how to measure the **true semantic similarity** between two sentences.

### Problem
Traditional similarity methods may consider `"I love chocolate"` and `"I hate chocolate"` as highly similar because the words are almost the same. But their **meaning is opposite**.

### Solution
We combine three techniques:
1. **Sentence Transformers** — convert sentences into embeddings (numerical vectors)
2. **Cosine Similarity** — measure topic similarity between embeddings
3. **Sentiment Analysis (VADER)** — detect if sentences have opposite feelings, and adjust the score accordingly

---

## Step 1 — Import Libraries

| Library | Purpose |
|---|---|
| `sentence_transformers` | Generate sentence embeddings using pretrained models |
| `sklearn` | Compute cosine similarity between vectors |
| `numpy` | Numerical operations |
| `vaderSentiment` | Sentiment analysis — detect positive/negative/neutral |


In [ ]:
# Import required libraries
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import numpy as np

print("All libraries imported successfully!")

## Step 2 — Load Pretrained Sentence Transformer Model

We use **`all-MiniLM-L6-v2`** — a lightweight pretrained model.

**What it does:**
- Converts any sentence into a **384-dimensional numerical vector** (embedding)
- Embeddings capture the **semantic meaning** of the sentence
- Already trained on millions of sentence pairs — no training needed by us

In [ ]:
# Load the pretrained Sentence Transformer model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print("Sentence Transformer model loaded!")
print("Model: all-MiniLM-L6-v2")
print("Embedding dimension: 384")

## Step 3 — Generate Sentence Embeddings

Let's take two sentences that look similar in words but have **opposite meaning**:

| Sentence | Meaning |
|---|---|
| "I love chocolate" | Positive feeling |
| "I hate chocolate" | Negative feeling |

We convert them into embeddings (numerical vectors) using the model.

In [ ]:
# Define two example sentences with opposite sentiment
sentence1 = "I love chocolate"
sentence2 = "I hate chocolate"

print(f"Sentence 1: {sentence1}")
print(f"Sentence 2: {sentence2}")
print()

# Generate embeddings
embeddings = model.encode([sentence1, sentence2])

print(f"Embedding shape: {embeddings.shape}")
print(f"  → 2 sentences, each as a 384-dimensional vector")
print(f"\nEmbedding for Sentence 1 (first 10 values): {np.round(embeddings[0][:10], 4)}")
print(f"Embedding for Sentence 2 (first 10 values): {np.round(embeddings[1][:10], 4)}")

## Step 4 — Calculate Cosine Similarity

**Cosine Similarity** measures the angle between two vectors.

| Score | Meaning |
|---|---|
| **1** | Very similar |
| **0** | Unrelated |
| **-1** | Opposite |

⚠️ **The problem:** Cosine similarity only measures **topic** similarity. It doesn't understand **emotional meaning**. So "I love X" and "I hate X" may get a high score because they talk about the same topic.

In [ ]:
# Compute Cosine Similarity
similarity = cosine_similarity(
    [embeddings[0]],  # Embedding of Sentence 1
    [embeddings[1]]   # Embedding of Sentence 2
)

original_score = round(float(similarity[0][0]), 4)

print(f"Cosine Similarity Score: {original_score}")
print()
print("⚠️ This score is HIGH because both sentences talk about 'chocolate'.")
print("   But their meanings are OPPOSITE (love vs hate).")
print("   This is where Sentiment Analysis helps!")

## Step 5 — Sentiment Analysis with VADER

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) analyzes the sentiment of a sentence.

It returns a **compound score**:
- `+1` → most positive
- `0` → neutral
- `-1` → most negative

**Why use it?** To detect if two sentences have **opposite emotional meaning** even though they talk about the same topic.

In [ ]:
# Initialize VADER Sentiment Analyzer
analyzer = SentimentIntensityAnalyzer()

# Get sentiment scores for both sentences
sentiment1 = analyzer.polarity_scores(sentence1)
sentiment2 = analyzer.polarity_scores(sentence2)

print(f"Sentence 1: '{sentence1}'")
print(f"  Sentiment scores: {sentiment1}")
print(f"  Compound score: {sentiment1['compound']} → {'Positive ✅' if sentiment1['compound'] > 0 else 'Negative ❌'}")
print()
print(f"Sentence 2: '{sentence2}'")
print(f"  Sentiment scores: {sentiment2}")
print(f"  Compound score: {sentiment2['compound']} → {'Positive ✅' if sentiment2['compound'] > 0 else 'Negative ❌'}")
print()
print("The sentiments are OPPOSITE — one is positive, the other is negative.")

## Step 6 — Adjust Similarity Using Sentiment

**Logic:**
- If `sentiment1 × sentiment2 < 0` → the sentiments are **opposite** (one positive, one negative)
- In that case, **reduce the similarity score** by multiplying it by `0.2`

This prevents sentences like "I love X" and "I hate X" from being considered highly similar.

In [ ]:
# Extract compound scores
compound1 = sentiment1['compound']
compound2 = sentiment2['compound']

# Check if sentiments are opposite
final_score = original_score

if compound1 * compound2 < 0:
    # Opposite sentiments — reduce similarity
    final_score = round(original_score * 0.2, 4)
    print(f"compound1 × compound2 = {round(compound1 * compound2, 4)} (NEGATIVE → opposite sentiments)")
    print(f"\n⚠️ Sentiment Adjustment Applied!")
    print(f"Original Score: {original_score} × 0.2 = {final_score}")
else:
    print(f"compound1 × compound2 = {round(compound1 * compound2, 4)} (POSITIVE → same sentiment)")
    print(f"No adjustment needed.")

## Step 7 — Display Final Result

In [ ]:
# Display the final result
print("=" * 55)
print("       SEMANTIC SIMILARITY RESULT")
print("=" * 55)
print(f"Sentence 1: {sentence1}")
print(f"Sentence 2: {sentence2}")
print(f"\nSentiment 1: {compound1} ({'Positive' if compound1 > 0 else 'Negative'})")
print(f"Sentiment 2: {compound2} ({'Positive' if compound2 > 0 else 'Negative'})")
print(f"\nOriginal Similarity: {original_score}")

if compound1 * compound2 < 0:
    print(f"\n⚠️ Sentiment Adjustment Applied")

print(f"\n✅ Final Similarity Score: {final_score}")
print()
print("Meaning: The sentences talk about the same topic")
print("         but have OPPOSITE sentiment.")
print("=" * 55)

## Bonus — Test More Sentence Pairs

Let's test with different scenarios to see how the system handles them.

In [ ]:
def compute_adjusted_similarity(s1, s2):
    """Helper function to compute sentiment-adjusted similarity."""
    emb = model.encode([s1, s2])
    cos_sim = float(cosine_similarity([emb[0]], [emb[1]])[0][0])
    sent1 = analyzer.polarity_scores(s1)['compound']
    sent2 = analyzer.polarity_scores(s2)['compound']
    
    adjusted = False
    final = cos_sim
    if sent1 * sent2 < 0:
        final = cos_sim * 0.2
        adjusted = True
    
    return round(cos_sim, 4), round(final, 4), round(sent1, 4), round(sent2, 4), adjusted

# Test cases
test_pairs = [
    ("I love chocolate", "I hate chocolate"),
    ("I love machine learning", "I enjoy studying artificial intelligence"),
    ("The movie was amazing", "The movie was terrible"),
    ("It is raining outside", "The weather is sunny and warm"),
    ("Python is great for data science", "Python is great for data science"),
]

print("Sentiment-Adjusted Similarity Scores:")
print("=" * 70)

for s1, s2 in test_pairs:
    orig, final, sent1, sent2, adj = compute_adjusted_similarity(s1, s2)
    print(f"\nSentence 1: {s1}")
    print(f"Sentence 2: {s2}")
    print(f"Sentiment: {sent1} / {sent2} | Original: {orig} | Final: {final}", end="")
    if adj:
        print(" ⚠️ ADJUSTED")
    else:
        print()
    print("-" * 70)

---

## Summary

In this notebook, we:

1. **Loaded** a pretrained Sentence Transformer model (`all-MiniLM-L6-v2`)
2. **Generated embeddings** for sentences
3. **Computed Cosine Similarity** to measure topic similarity
4. **Performed Sentiment Analysis** using VADER to detect positive/negative feelings
5. **Adjusted the similarity score** when sentiments are opposite

### Why is this important?

Without sentiment adjustment, sentences like *"I love chocolate"* and *"I hate chocolate"* would be considered ~80% similar because they use similar words. But their **meaning is opposite**.

By adding sentiment analysis, we reduce the score to reflect the **true semantic difference**.